Two reuse mechanisms: **how much extra engagement (views, likes, comments) does a derived video collect, relative to the original footage it came from?**

1. **Clipping gains** (PP *and* NM): source = a channel's own **RV**; derivatives = the
   Shorts / TikToks cut from it. Same channel on both sides. Edges from `gs_pp_only` /
   `gs_news_only`.
2. **Repost gains** (PP only): source = the politician's **NM appearance** (canonical NM
   RV); derivative = the politician's repost on their own channel. Edges from `gs_diff`,
   restricted to `reuse_direction == 'nm_to_pp'` (built exactly as in nb09).

**Design decisions :**

- **Per-clip *and* aggregated-to-source.** One RV can spawn several clips. `pairs_*` holds
  one row per (source, derivative). `by_source_*` sums the derivatives of each source, so
  "additional views from clipping RV *x*" = (sum of its clips' views) − (RV's views).
- **Absolute gain *and* ratio.** Absolute gain (`gain_*`) is dominated by a few viral
  videos and does not average across channels; the **amplification ratio**
  (`derivative / source`) is scale-free and is what we aggregate (by **median**, since it
  is right-skewed). Both are computed; reporting defaults to the ratio.
- **Platform is not silently mixed.** A YouTube "view" and a TikTok "view" are different
  metrics, and an RV↔TikTok comparison crosses them. Every pair carries
  `source_platform` / `derivative_platform`; clipping gains are reported **split by
  derivative platform** (YT-clip vs TT-clip).

# 0. Setup & imports

In [19]:
import os
import networkx as nx

if os.path.isdir("../data/"):
    os.chdir("../")

%load_ext autoreload
%autoreload 2

from src.utils import *
from notebooks.plots_utils import *
from scripts.project_config import get_party_orientation, get_politician2party

politician2party   = get_politician2party()
party_colors       = get_parties_color_maps()
party_orientation  = get_party_orientation()


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# 1. Configuration — engagement column names & options

**IMPORTANT / VERIFY.** The existing notebooks (08, 09) never touch engagement fields, so
I could not confirm from your code how views/likes/comments are named in your JSONL. The
defaults below follow the raw API conventions (YouTube API v3 `statistics.*`, TikTok
Research API `*_count`). **Run the schema check in the next cell**; if a name is wrong it
will list the columns you actually have so you can fix `ENGAGEMENT_COLS` here.

In [20]:
# --- Which raw column holds each metric, per platform. EDIT if the check below fails. ---
ENGAGEMENT_COLS = {
    'yt': {'views': 'viewCount',    'likes': 'likeCount',    'comments': 'commentCount'},
    'tt': {'views': 'view_count',   'likes': 'like_count',   'comments': 'comment_count'},
}
METRICS = ['views', 'likes', 'comments']   # drop 'comments' if too sparse; add 'shares' for TT if present

# Ratio hygiene: sources with 0 (or missing) on a metric make the ratio undefined -> dropped
# from that metric's ratio aggregation (kept for absolute gain). Set a floor if you prefer.
MIN_SOURCE_FOR_RATIO = 1     # require source metric >= this to compute a ratio
CLIP_MIN_PAIRS       = 3     # per-channel: min matched pairs to report a channel-level number


# 2. Load videos (YT + TT, both periods) and build engagement lookups

Same loading as nb08/nb09. We keep 2022 + 2024 so pairs are resolvable whichever period a video is from; filter by `publishedAt` later if you want 2024 only.

In [21]:
# ---- YouTube ----
yt_nm_2022 = pd.read_json("data/youtube/videos/news_videos_2022.jsonl", lines=True)
yt_nm_2024 = pd.read_json("data/youtube/videos/news_videos_2024.jsonl", lines=True)
yt_pp_2022 = pd.read_json("data/youtube/videos/pp_videos_2022.jsonl",   lines=True)
yt_pp_2024 = pd.read_json("data/youtube/videos/pp_videos_2024.jsonl",   lines=True)
YT_DFS = [yt_nm_2022, yt_nm_2024, yt_pp_2022, yt_pp_2024]
for _df in YT_DFS:
    _df['publishedAt'] = pd.to_datetime(_df['publishedAt']).dt.tz_localize(None)
    _df.rename(columns = {'views_2025-04':"viewCount", 'likes_2025-04':"likeCount", 'comments_2025-04':"commentCount",
                          'views_2025-01':"viewCount", 'likes_2025-01':"likeCount", 'comments_2025-01':"commentCount",
                             'views_2025-08':"viewCount", 'likes_2025-08':"likeCount", 'comments_2025-08':"commentCount"}, inplace=True)

# ---- TikTok ----
tt_nm_2022 = pd.read_json("data/tiktok/videos/news_videos_2022.jsonl", lines=True)
tt_nm_2024 = pd.read_json("data/tiktok/videos/news_videos_2024.jsonl", lines=True)
tt_pp_2022 = pd.read_json("data/tiktok/videos/pp_videos_2022.jsonl",   lines=True)
tt_pp_2024 = pd.read_json("data/tiktok/videos/pp_videos_2024.jsonl",   lines=True)
TT_DFS = [tt_nm_2022, tt_nm_2024, tt_pp_2022, tt_pp_2024]
for _df in TT_DFS:
    _df.rename(columns={'id': 'videoId', 'voice_to_text': 'transcript',
                        'create_time': 'publishedAt'}, inplace=True)
    _df['videoId'] = _df['videoId'].apply(str)
    _df['publishedAt'] = pd.to_datetime(_df['publishedAt']).dt.tz_localize(None)


In [22]:
# ---- Unified lookups (mirror nb09) ----
PLATFORM_CODE = {"youtube": "yt", "tiktok": "tt"}

def get_platform_from_videoId(videoid):
    return PLATFORM_CODE[get_platform(videoid)]  # get_platform: src/utils.py

videoId2publishedAt = {}
videoid2standard_name = {}
videoId2yt_type = {}     # 'RV'/'Short' for YT, 'TikTok' for TT

for _df in YT_DFS + TT_DFS:
    videoId2publishedAt.update(dict(zip(_df['videoId'], _df['publishedAt'])))
    videoid2standard_name.update(dict(zip(_df['videoId'], _df['name_standard'])))
for _df in YT_DFS:
    videoId2yt_type.update(dict(zip(_df['videoId'],
                             ['Short' if e else 'RV' for e in _df['isShort']])))
for _df in TT_DFS:
    videoId2yt_type.update({vid: 'TikTok' for vid in _df['videoId']})

# ---- videoId -> {metric: value}, coerced to numeric, per platform ----
def _engagement_lookup(dfs, plat):
    colmap = ENGAGEMENT_COLS[plat]
    out = {}
    for _df in dfs:
        sub = _df[['videoId'] + list(colmap.values())].copy()
        for m, c in colmap.items():
            sub[m] = pd.to_numeric(sub[c], errors='coerce')
        for r in sub.itertuples(index=False):
            out[getattr(r, 'videoId')] = {m: getattr(r, m) for m in colmap}
    return out

eng_yt = _engagement_lookup(YT_DFS, 'yt')
eng_tt = _engagement_lookup(TT_DFS, 'tt')

def get_engagement(videoId, metric):
    v = str(videoId)
    d = eng_yt.get(v) or eng_tt.get(v)
    return np.nan if d is None else d.get(metric, np.nan)

print(f"engagement known for {len(eng_yt)} YT + {len(eng_tt)} TT videos")


engagement known for 80322 YT + 13462 TT videos


# 3. Load graphs + channel lists (as in nb08/nb09)

In [23]:
def load_gs(filepath):
    filenames = [f for f in os.listdir(filepath) if f.endswith('.gv')]
    return {f.split('_')[0]: nx.nx_agraph.read_dot(filepath + f) for f in filenames}

gs_pp_only   = load_gs('data/networks/pp_only_2024/')
gs_news_only = load_gs('data/networks/news_only_2024/')
gs_diff      = load_gs('data/networks/diff_2024/')

nm_yt_channels = pd.read_json('data/youtube/channels/news_channels.json')
pp_yt_channels = pd.read_json('data/youtube/channels/pp_channels.json')
nm_name2orientation = dict(zip(nm_yt_channels.name_standard, nm_yt_channels.orientation))

stat_not_significant_channels = ['Gabriel Attal', 'LP', 'LR', 'RE', 'Rima Hassan', 'UDI']
yt_pp_standard_names = list(set(pp_yt_channels.name_standard) - set(stat_not_significant_channels))
yt_nm_standard_names = list(nm_yt_channels.name_standard)


# 4. Clipping gains (PP and NM)

For each channel graph, every connected component is one source RV plus the short-form
clips cut from it (same convention nb08 uses via `count_distinct_cuts`). We emit one
**pair** per (RV, clip). A component with no RV node (only a Short/TikTok, no captured
long-form) has no measurable source and is skipped — recorded in `n_components_no_rv`.

In [24]:
def clipping_pairs(gs, name_standard_list):
    """One row per (source RV, derived clip) for every channel in the list."""
    rows, skipped = [], {}
    for name in name_standard_list:
        if name not in gs:
            continue
        g = gs[name].to_undirected()
        no_rv = 0
        for comp in nx.connected_components(g):
            rv_nodes = [n for n in comp if videoId2yt_type.get(n) == 'RV']
            if not rv_nodes:
                no_rv += 1
                continue
            src = rv_nodes[0]                     # canonical source = the RV
            clips = [n for n in comp if n != src]
            for c in clips:
                rows.append({
                    'name_standard': name,
                    'party': politician2party.get(name),
                    'source': src,
                    'derivative': c,
                    'source_type': videoId2yt_type.get(src),
                    'derivative_type': videoId2yt_type.get(c),
                    'source_platform': get_platform_from_videoId(src),
                    'derivative_platform': get_platform_from_videoId(c),
                })
        skipped[name] = no_rv
    df = pd.DataFrame(rows)
    return df, skipped


def add_engagement(df, metrics=METRICS):
    """Attach source/derivative metric values, absolute gain, and amplification ratio."""
    df = df.copy()
    for m in metrics:
        df[f'src_{m}'] = df['source'].map(lambda v: get_engagement(v, m))
        df[f'der_{m}'] = df['derivative'].map(lambda v: get_engagement(v, m))
        df[f'gain_{m}'] = df[f'der_{m}'] - df[f'src_{m}']
        denom = df[f'src_{m}'].where(df[f'src_{m}'] >= MIN_SOURCE_FOR_RATIO)
        df[f'ratio_{m}'] = df[f'der_{m}'] / denom
    return df


In [25]:
pairs_pp_clip, skip_pp = clipping_pairs(gs_pp_only,   yt_pp_standard_names)
pairs_nm_clip, skip_nm = clipping_pairs(gs_news_only, yt_nm_standard_names)
pairs_pp_clip = add_engagement(pairs_pp_clip)
pairs_nm_clip = add_engagement(pairs_nm_clip)

print(f"PP clip pairs: {len(pairs_pp_clip)}  |  NM clip pairs: {len(pairs_nm_clip)}")
print("components with no RV source (PP):", sum(skip_pp.values()),
      "| (NM):", sum(skip_nm.values()))
pairs_pp_clip.head()


PP clip pairs: 863  |  NM clip pairs: 4936
components with no RV source (PP): 1255 | (NM): 4604


,name_standard,party,source,derivative,source_type,derivative_type,source_platform,derivative_platform,src_views,der_views,gain_views,ratio_views,src_likes,der_likes,gain_likes,ratio_likes,src_comments,der_comments,gain_comments,ratio_comments
0,Manon Aubry,LFI,K2jDii0oGFY,7356174792950566177,RV,TikTok,yt,tt,2586,3164,578,1.223511,476.0,283.0,-193.0,0.594538,NaN,40.0,NaN,NaN
1,Manon Aubry,LFI,K2jDii0oGFY,7356633290540911904,RV,TikTok,yt,tt,2586,142613,140027,55.148105,476.0,9275.0,8799.0,19.485294,NaN,764.0,NaN,NaN
2,Manon Aubry,LFI,K2jDii0oGFY,I__CVDkSsMk,RV,Short,yt,yt,2586,119785,117199,46.320572,476.0,4886.0,4410.0,10.264706,NaN,414.0,NaN,NaN
3,Manon Aubry,LFI,K2jDii0oGFY,JlNnObTq2SA,RV,RV,yt,yt,2586,7901,5315,3.055298,476.0,932.0,456.0,1.957983,NaN,242.0,NaN,NaN
4,Manon Aubry,LFI,RYQdrpY8awc,p92WGKIwQug,RV,RV,yt,yt,5475,3910,-1565,0.714155,665.0,495.0,-170.0,0.744361,210.0,59.0,-151.0,0.280952


In [26]:
# ---- Aggregate clips to their source RV: 'additional engagement from clipping this RV' ----
def by_source(pairs, metrics=METRICS):
    g = pairs.groupby(['name_standard', 'party', 'source', 'source_platform'])
    agg = g.agg(
        n_clips=('derivative', 'size'),
        der=('derivative', lambda s: '|'.join(map(str, sorted(set(s))))),
    ).reset_index()
    for m in metrics:
        src_first = g[f'src_{m}'].first().reset_index(drop=True)
        sum_der   = g[f'der_{m}'].sum(min_count=1).reset_index(drop=True)
        agg[f'src_{m}']       = src_first
        agg[f'sum_clip_{m}']  = sum_der
        agg[f'total_gain_{m}'] = agg[f'sum_clip_{m}'] - agg[f'src_{m}']
        denom = agg[f'src_{m}'].where(agg[f'src_{m}'] >= MIN_SOURCE_FOR_RATIO)
        agg[f'amplification_{m}'] = agg[f'sum_clip_{m}'] / denom
    return agg

by_source_pp = by_source(pairs_pp_clip)
by_source_nm = by_source(pairs_nm_clip)
by_source_pp.head()


,name_standard,party,source,source_platform,n_clips,der,src_views,sum_clip_views,total_gain_views,amplification_views,src_likes,sum_clip_likes,total_gain_likes,amplification_likes,src_comments,sum_clip_comments,total_gain_comments,amplification_comments
0,EELV,EELV,3wWtPw6DfsE,yt,12,-sAnOXntTCk|-uXhs4SIlSY|7389014072567631136|Hb...,2131,171929,169798,80.679962,91.0,6443.0,6352.0,70.802198,6.0,2337.0,2331.0,389.500000
1,EELV,EELV,AytMAraIsN0,yt,1,7385432419312209184,13969,3615,-10354,0.258787,465.0,517.0,52.0,1.111828,98.0,11.0,-87.0,0.112245
2,EELV,EELV,GnjmEcxk7Yc,yt,1,7385613061534633248,9423,1440,-7983,0.152818,324.0,130.0,-194.0,0.401235,257.0,11.0,-246.0,0.042802
3,EELV,EELV,T7LQXzEcsXY,yt,1,7377799124764101920,1445,983,-462,0.680277,40.0,60.0,20.0,1.500000,24.0,4.0,-20.0,0.166667
4,EELV,EELV,fj8VHEzuPGk,yt,2,7388259253515013408|Gw7QWICElZM,2504,3873,1369,1.546725,201.0,365.0,164.0,1.815920,38.0,31.0,-7.0,0.815789


### 4a. Clipping gains — summary tables

Reported as the **median amplification ratio** (derivative engagement as a multiple of the
source's own), which is the scale-free, skew-robust number. `>1` means the reuse collected
more than the original; `<1` means less. Split by **derivative platform** so YouTube-view
and TikTok-view comparisons are not blended, and the pure same-metric YT→YT case is
isolated.

In [27]:
def clipping_summary(pairs, label, metrics=METRICS):
    """Per-derivative-platform medians of the amplification ratio + per-clip absolute gain."""
    out = []
    for der_plat, sub in pairs.groupby('derivative_platform'):
        clean = 'same-metric (YT RV -> YT clip)' if der_plat == 'yt' else 'cross-metric (YT RV -> TT clip)'
        row = {'set': label, 'derivative_platform': der_plat, 'comparison': clean,
               'n_pairs': len(sub)}
        for m in metrics:
            row[f'median_ratio_{m}'] = sub[f'ratio_{m}'].median()
            row[f'median_gain_{m}']  = sub[f'gain_{m}'].median()
            row[f'share_amplified_{m}'] = (sub[f'ratio_{m}'] > 1).mean()   # fraction of clips that beat their RV
        out.append(row)
    return pd.DataFrame(out)

clip_summary = pd.concat([
    clipping_summary(pairs_pp_clip, 'PP'),
    clipping_summary(pairs_nm_clip, 'NM'),
], ignore_index=True)
clip_summary.round(3)


,set,derivative_platform,comparison,n_pairs,median_ratio_views,median_gain_views,share_amplified_views,median_ratio_likes,median_gain_likes,share_amplified_likes,median_ratio_comments,median_gain_comments,share_amplified_comments
0,PP,tt,cross-metric (YT RV -> TT clip),511,1.268,2666.0,0.554,1.406,308.0,0.528,0.382,-180.0,0.315
1,PP,yt,same-metric (YT RV -> YT clip),352,1.978,12368.5,0.685,2.282,1219.0,0.716,0.761,-34.5,0.420
2,NM,tt,cross-metric (YT RV -> TT clip),2965,1.398,1844.5,0.515,3.353,186.0,0.619,0.400,-12.0,0.284
3,NM,yt,same-metric (YT RV -> YT clip),1971,0.959,-386.0,0.460,1.430,32.0,0.553,0.488,-12.0,0.303


# 5. Repost gains (PP only): NM appearance -> politician repost

Rebuild the canonical NM-RV collapse and the deduplicated PP↔NM edge table exactly as in
nb09, then keep only genuine `nm_to_pp` reposts. **Source = the NM appearance's canonical
RV; derivative = the PP repost.** Because a politician may clip a reposted appearance
further, we can measure the gain at two source→target granularities (see 5b).

In [28]:
# ---- canonical NM RV map (identical logic to nb09 cell) ----
def _earliest_node(nodes):
    return min(nodes, key=lambda n: videoId2publishedAt.get(n, pd.Timestamp.max))

nm_clip2rv = {}
for nm_graph in gs_news_only.values():
    g = nm_graph.to_undirected()
    for component in nx.connected_components(g):
        rvs = [n for n in component if videoId2yt_type.get(n) == 'RV']
        canonical = rvs[0] if rvs else _earliest_node(component)
        for n in component:
            nm_clip2rv[n] = canonical
print(f"{len(nm_clip2rv)} NM nodes mapped to canonical video")


40265 NM nodes mapped to canonical video


In [29]:
# ---- PP<->NM edges, canonical + deduplicated (nb09 cell 28) ----
def nm_repost_edges(name_standard, gs_diff):
    if name_standard not in gs_diff:
        return pd.DataFrame()
    g = gs_diff[name_standard]
    ntype = nx.get_node_attributes(g, 'type')
    rows = []
    for u, v in g.edges():
        tu, tv = ntype.get(u, ''), ntype.get(v, '')
        if tv == 'nm' and tu in ('youtube_pp', 'tiktok_pp'):
            pp_id, nm_id = u, v
        elif tu == 'nm' and tv in ('youtube_pp', 'tiktok_pp'):
            pp_id, nm_id = v, u
        else:
            continue
        nm_rv = nm_clip2rv.get(nm_id, nm_id)
        pp_pub, rv_pub = videoId2publishedAt.get(pp_id), videoId2publishedAt.get(nm_rv)
        elapsed = direction = None
        if pp_pub is not None and rv_pub is not None:
            elapsed = (pp_pub - rv_pub).total_seconds()
            direction = 'nm_first' if rv_pub < pp_pub else 'pp_first'
        rows.append({'name_standard': name_standard, 'party': politician2party.get(name_standard),
                     'pp_videoId': pp_id, 'pp_platform': get_platform_from_videoId(pp_id),
                     'nm_videoId': nm_rv, 'nm_outlet': videoid2standard_name.get(nm_rv),
                     'nm_orientation': None, 'elapsed_sec': elapsed, 'direction': direction})
    df = pd.DataFrame(rows)
    if not len(df):
        return df
    df = df.sort_values('elapsed_sec').drop_duplicates(['pp_videoId', 'nm_videoId'], keep='first')
    n_channels = df.groupby('pp_videoId')['nm_outlet'].transform('nunique')
    df['reuse_direction'] = np.where(n_channels > 1, 'pp_to_nm', 'nm_to_pp')
    df = df.sort_values('elapsed_sec')
    first = df.groupby(['name_standard', 'nm_videoId']).head(1).index
    df['is_pp_reclip'] = ~df.index.isin(first)
    df['nm_orientation'] = df['nm_outlet'].map(nm_name2orientation)
    return df

edges_all = pd.concat([nm_repost_edges(n, gs_diff) for n in yt_pp_standard_names],
                      ignore_index=True)
print("total edges:", len(edges_all))
print(edges_all['reuse_direction'].value_counts(dropna=False))


total edges: 1354
reuse_direction
pp_to_nm    766
nm_to_pp    588
Name: count, dtype: int64


### 5a. Gain of the direct repost vs the NM appearance

Keep genuine `nm_to_pp` reposts. Source metric = the NM appearance (canonical RV);
derivative metric = the politician's repost. Same absolute-gain + amplification-ratio
scheme as clipping. `pp_platform` records whether the repost landed on YouTube or TikTok,
so the cross-metric caveat applies identically.

In [30]:
reposts = edges_all[edges_all['reuse_direction'] == 'nm_to_pp'].copy()

for m in METRICS:
    reposts[f'src_{m}'] = reposts['nm_videoId'].map(lambda v: get_engagement(v, m))   # NM appearance
    reposts[f'der_{m}'] = reposts['pp_videoId'].map(lambda v: get_engagement(v, m))   # PP repost
    reposts[f'gain_{m}'] = reposts[f'der_{m}'] - reposts[f'src_{m}']
    denom = reposts[f'src_{m}'].where(reposts[f'src_{m}'] >= MIN_SOURCE_FOR_RATIO)
    reposts[f'ratio_{m}'] = reposts[f'der_{m}'] / denom

reposts['elapsed_days'] = reposts['elapsed_sec'] / 86400

def repost_summary(df, metrics=METRICS):
    out = []
    for plat, sub in df.groupby('pp_platform'):
        row = {'repost_platform': plat, 'n_pairs': len(sub)}
        for m in metrics:
            row[f'median_ratio_{m}']    = sub[f'ratio_{m}'].median()
            row[f'median_gain_{m}']     = sub[f'gain_{m}'].median()
            row[f'share_amplified_{m}'] = (sub[f'ratio_{m}'] > 1).mean()
        out.append(row)
    return pd.DataFrame(out)

repost_summary(reposts).round(3)


,repost_platform,n_pairs,median_ratio_views,median_gain_views,share_amplified_views,median_ratio_likes,median_gain_likes,share_amplified_likes,median_ratio_comments,median_gain_comments,share_amplified_comments
0,tt,203,0.789,-2941.5,0.429,3.951,1572.0,0.704,0.267,-361.5,0.256
1,yt,385,0.744,-2414.0,0.403,3.046,677.0,0.634,0.697,-45.0,0.384


### 5b. (Optional) Total reach of an appearance, clips included

A reposted appearance is sometimes clipped again by the politician. To capture the *full*
additional engagement an appearance generated on the PP side, sum the repost **and** any
of its downstream clips, then compare to the NM appearance. Uses `gs_pp_only` to find the
clips of each repost video. Left as an optional extension — the direct measure in 5a is the
clean one; this one re-introduces cross-platform metric mixing.

In [31]:
# Map each PP video to the clips cut from it (within the politician's own graph).
def pp_clip_children(name_standard):
    if name_standard not in gs_pp_only:
        return {}
    g = gs_pp_only[name_standard].to_undirected()
    children = {}
    for comp in nx.connected_components(g):
        rv = [n for n in comp if videoId2yt_type.get(n) == 'RV']
        if not rv:
            continue
        src = rv[0]
        children[src] = [n for n in comp if n != src]
    return children

def repost_total_reach(df, metrics=METRICS):
    df = df.copy()
    kids_by_chan = {n: pp_clip_children(n) for n in df['name_standard'].unique()}
    for m in metrics:
        tot = []
        for r in df.itertuples(index=False):
            base = get_engagement(r.pp_videoId, m)
            kids = kids_by_chan.get(r.name_standard, {}).get(r.pp_videoId, [])
            base = 0 if pd.isna(base) else base
            tot.append(base + sum(np.nan_to_num(get_engagement(k, m)) for k in kids))
        df[f'total_pp_{m}'] = tot
        denom = df[f'src_{m}'].where(df[f'src_{m}'] >= MIN_SOURCE_FOR_RATIO)
        df[f'total_amplification_{m}'] = df[f'total_pp_{m}'] / denom
    return df

reposts_total = repost_total_reach(reposts)
reposts_total[['name_standard', 'nm_videoId', 'pp_videoId'] +
              [f'total_amplification_{m}' for m in METRICS]].head()


,name_standard,nm_videoId,pp_videoId,total_amplification_views,total_amplification_likes,total_amplification_comments
6,Manon Aubry,7356149830365793569,1_BmaCvV3VY,1.067742,3.334728,4.684211
7,Manon Aubry,IKQ8NCJqI6s,gOuUnN8hy0M,6.868541,59.596774,10.586207
8,Manon Aubry,E8iqI_7F6-M,7374903943295470881,0.067137,0.385124,0.031805
9,Manon Aubry,6hMhr5xYuVc,frUdd1legTM,0.667787,1.422840,1.190114
11,Manon Aubry,QZKeUfn2wKI,TvreVzntKjs,2.412389,4.703043,2.530120


### 4b. Clipping gains — per channel and per party

Per-channel amplification (median across that channel's clips), reported only for channels
with at least `CLIP_MIN_PAIRS` matched pairs; then averaged to party level. Restricted to
**YT→YT** here for a clean same-metric read — flip `DER_PLATFORM` to `'tt'` for the TikTok
side, or `None` for both.

In [37]:
def median_ratio_views(pairs, platform_filter=None):
    """Per-channel median of per-pair views ratio, optionally filtered by derivative platform."""
    sub = pairs if platform_filter is None else pairs[pairs['derivative_platform'] == platform_filter]
    g = sub.groupby(['name_standard', 'party'])
    out = g['ratio_views'].agg(['median', 'size']).reset_index()
    out = out[out['size'] >= CLIP_MIN_PAIRS]          # same support threshold as before
    return out.set_index(['name_standard', 'party'])['median']

# clip amplification, split by where the clip landed
yt_short = median_ratio_views(pairs_pp_clip, 'yt').rename('views_via_shorts')
tiktok   = median_ratio_views(pairs_pp_clip, 'tt').rename('views_via_tiktok')

# repost amplification (PP repost views vs NM appearance views); reposts has no
# 'derivative_platform' col, so filter on pp_platform=None => all reposts
rep_g = reposts.groupby(['name_standard', 'party'])
repost = rep_g['ratio_views'].agg(['median', 'size'])
repost = repost[repost['size'] >= CLIP_MIN_PAIRS]['median'].rename('views_via_reposts')

chan_views = (pd.concat([yt_short, tiktok, repost], axis=1)
                .reset_index())
chan_views.round(2)

,name_standard,party,views_via_shorts,views_via_tiktok,views_via_reposts
0,EELV,EELV,6.29,0.30,0.33
1,Emmanuel Macron,RE,2.60,112.90,4.22
2,François Ruffin,LFI,0.49,0.27,0.87
3,Jean-Luc Mélenchon,LFI,1.59,1.65,3.44
4,Jordan Bardella,RN,2.53,18.83,1.87
5,LFI,LFI,3.41,NaN,1.38
6,Manon Aubry,LFI,0.87,2.05,1.08
7,Manuel Bompard,LFI,1.67,0.74,2.38
8,Marine Le Pen,RN,6.72,21.43,0.61
9,Marion Maréchal,REC,0.84,1.04,1.02


In [38]:
# channels that survived into the table, per party
kept = chan_views.dropna().set_index('name_standard')['party'].to_dict()

def pair_counts(pairs, platform_filter=None):
    sub = pairs if platform_filter is None else pairs[pairs['derivative_platform'] == platform_filter]
    sub = sub[sub['name_standard'].isin(kept)]
    return sub.groupby('name_standard').size().groupby(kept).sum()  # -> per party

n_shorts  = pair_counts(pairs_pp_clip, 'yt').rename('n_shorts')
n_tiktok  = pair_counts(pairs_pp_clip, 'tt').rename('n_tiktok')
n_repost  = (reposts[reposts['name_standard'].isin(kept)]
             .groupby('name_standard').size().groupby(kept).sum().rename('n_repost'))

party_views = (chan_views.dropna()
    .groupby('party')[['views_via_shorts', 'views_via_tiktok', 'views_via_reposts']]
    .mean()
    .assign(n_channels=chan_views.dropna().groupby('party').size()))

party_views = party_views.join([n_shorts, n_tiktok, n_repost]).fillna(0)


DAGGER_THRESHOLD = 24

def fmt(val, n):
    mark = "$^{\\dagger}$" if n < DAGGER_THRESHOLD else ""
    return f"{val:.2f}{mark}"

order = ['views_via_shorts', 'n_shorts', 'views_via_tiktok', 'n_tiktok',
         'views_via_reposts', 'n_repost']
for party, r in party_views[order].iterrows():
    print(f"{party} & {fmt(r['views_via_shorts'], r['n_shorts'])} & {int(r['n_shorts'])} & "
          f"{fmt(r['views_via_tiktok'], r['n_tiktok'])} & {int(r['n_tiktok'])} & "
          f"{fmt(r['views_via_reposts'], r['n_repost'])} & {int(r['n_repost'])} \\\\")

EELV & 6.29$^{\dagger}$ & 12 & 0.30$^{\dagger}$ & 11 & 0.33$^{\dagger}$ & 21 \\
LFI & 1.56 & 136 & 2.02 & 201 & 2.14 & 165 \\
PCF & 2.42 & 102 & 0.60 & 58 & 1.15 & 64 \\
RE & 2.60$^{\dagger}$ & 4 & 112.90$^{\dagger}$ & 14 & 4.22$^{\dagger}$ & 5 \\
REC & 0.72 & 33 & 0.77 & 97 & 0.92 & 77 \\
RN & 4.62 & 55 & 20.13 & 57 & 1.24 & 80 \\


# why RE tiktok gains so high ? 

In [39]:
pairs_pp_clip[(pairs_pp_clip['party'] == 'RE') & (pairs_pp_clip['derivative_type']== 'TikTok')]

,name_standard,party,source,derivative,source_type,derivative_type,source_platform,derivative_platform,src_views,der_views,gain_views,ratio_views,src_likes,der_likes,gain_likes,ratio_likes,src_comments,der_comments,gain_comments,ratio_comments
472,Emmanuel Macron,RE,fslUZZ_i4DU,7367692331706797344,RV,TikTok,yt,tt,13525,4790263,4776738,354.178410,568.0,361178.0,360610.0,635.876761,273.0,10059.0,9786.0,36.846154
473,Emmanuel Macron,RE,fslUZZ_i4DU,7367651453185379616,RV,TikTok,yt,tt,13525,1772156,1758631,131.028170,568.0,139474.0,138906.0,245.552817,273.0,2188.0,1915.0,8.014652
474,Emmanuel Macron,RE,fslUZZ_i4DU,7367756877209357600,RV,TikTok,yt,tt,13525,5086822,5073297,376.105139,568.0,277685.0,277117.0,488.882042,273.0,14631.0,14358.0,53.593407
476,Emmanuel Macron,RE,fslUZZ_i4DU,7368046206871735584,RV,TikTok,yt,tt,13525,671209,657684,49.627283,568.0,27726.0,27158.0,48.813380,273.0,1175.0,902.0,4.304029
477,Emmanuel Macron,RE,fslUZZ_i4DU,7367675674632490272,RV,TikTok,yt,tt,13525,2614573,2601048,193.314085,568.0,155611.0,155043.0,273.963028,273.0,9680.0,9407.0,35.457875
478,Emmanuel Macron,RE,fslUZZ_i4DU,7368187420690500896,RV,TikTok,yt,tt,13525,1015501,1001976,75.083253,568.0,47156.0,46588.0,83.021127,273.0,3592.0,3319.0,13.157509
479,Emmanuel Macron,RE,fslUZZ_i4DU,7368015406138756384,RV,TikTok,yt,tt,13525,817867,804342,60.470758,568.0,40049.0,39481.0,70.508803,273.0,1166.0,893.0,4.271062
480,Emmanuel Macron,RE,fslUZZ_i4DU,7367735643960642849,RV,TikTok,yt,tt,13525,4699489,4685964,347.466839,568.0,327980.0,327412.0,577.429577,273.0,4752.0,4479.0,17.406593
481,Emmanuel Macron,RE,fslUZZ_i4DU,7367788304382692640,RV,TikTok,yt,tt,13525,1313527,1300002,97.118447,568.0,78935.0,78367.0,138.970070,273.0,2353.0,2080.0,8.619048
483,Emmanuel Macron,RE,fslUZZ_i4DU,7368119656382582049,RV,TikTok,yt,tt,13525,2642963,2629438,195.413161,568.0,203354.0,202786.0,358.017606,273.0,4723.0,4450.0,17.300366


In [40]:
pairs_pp_clip[(pairs_pp_clip['source'] == 'fslUZZ_i4DU') & (pairs_pp_clip['derivative_type']== 'TikTok')]['der_views'].sum()

np.int64(27164655)

In [41]:
257881/31960

8.06886733416771

In [42]:
1587971/98148

16.1793515914741

In [43]:
265318/12084

21.95614035087719

In [44]:
(21 + 8 + 16) / 3

15.0

# 6. Consolidated numbers for the paper

Median amplification ratios side by side. 

In [47]:
def headline(df, label, ratio_prefix='ratio', platform_col=None, metrics=METRICS):
    rows = []
    groups = [(label, df)] if platform_col is None else \
             [(f'{label} [{p}]', s) for p, s in df.groupby(platform_col)]
    for name, sub in groups:
        r = {'set': name, 'n_pairs': len(sub)}
        for m in metrics:
            col = f'{ratio_prefix}_{m}'
            r[f'median_ampl_{m}'] = sub[col].median()
            r[f'%_amplified_{m}'] = (sub[col] > 1).mean()
        rows.append(r)
    return pd.DataFrame(rows)

headline_tbl = pd.concat([
    headline(pairs_pp_clip, 'PP clip', platform_col='derivative_platform'),
    headline(pairs_nm_clip, 'NM clip', platform_col='derivative_platform'),
    headline(reposts,       'PP repost', platform_col='pp_platform'),
], ignore_index=True)
headline_tbl.round(3)


,set,n_pairs,median_ampl_views,%_amplified_views,median_ampl_likes,%_amplified_likes,median_ampl_comments,%_amplified_comments
0,PP clip [tt],511,1.268,0.554,1.406,0.528,0.382,0.315
1,PP clip [yt],352,1.978,0.685,2.282,0.716,0.761,0.420
2,NM clip [tt],2965,1.398,0.515,3.353,0.619,0.400,0.284
3,NM clip [yt],1971,0.959,0.460,1.430,0.553,0.488,0.303
4,PP repost [tt],203,0.789,0.429,3.951,0.704,0.267,0.256
5,PP repost [yt],385,0.744,0.403,3.046,0.634,0.697,0.384


In [48]:
# quick sanity / coverage report — how much of the matched reuse actually has engagement data
def coverage_report(df, s_ids, d_ids):
    s = df[s_ids].map(lambda v: not pd.isna(get_engagement(v, 'views')))
    d = df[d_ids].map(lambda v: not pd.isna(get_engagement(v, 'views')))
    return pd.Series({'n_pairs': len(df),
                      'source_has_views': s.mean(),
                      'derivative_has_views': d.mean(),
                      'both_have_views': (s & d).mean()})

print("PP clip:\n",   coverage_report(pairs_pp_clip, 'source', 'derivative'), "\n")
print("NM clip:\n",   coverage_report(pairs_nm_clip, 'source', 'derivative'), "\n")
print("PP repost:\n", coverage_report(reposts, 'nm_videoId', 'pp_videoId'))


PP clip:
 n_pairs                 863.0
source_has_views          1.0
derivative_has_views      1.0
both_have_views           1.0
dtype: float64 

NM clip:
 n_pairs                 4936.000000
source_has_views           0.930916
derivative_has_views       0.978120
both_have_views            0.930916
dtype: float64 

PP repost:
 n_pairs                 588.000000
source_has_views          0.914966
derivative_has_views      1.000000
both_have_views           0.914966
dtype: float64
